Load Models and Data

In [ ]:
# from google.colab import files
# files.upload()

In [ ]:
!pip install "torch>=1.3" "datasets==2.0.0" "sentencepiece!=0.1.92" "transformers==4.25.0" "nltk==3.6.5" protobuf tensorboardX wandb jieba nltk evaluate

In [ ]:
from __future__ import absolute_import
from __future__ import division

import os
import argparse
import traceback
from pathlib import Path

import socket
from collections import OrderedDict
from typing import *

import torch
import torch.nn as nn
import wandb
from tensorboardX import SummaryWriter
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from transformers import AdamW, get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup

from transformers import (
    MT5ForConditionalGeneration,
    MT5Tokenizer,
    MBartForConditionalGeneration,
    MBartTokenizer,
    MBart50Tokenizer,
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    AutoModelForSeq2SeqLM, # for indicBART
    AlbertTokenizer, #https://huggingface.co/ai4bharat/IndicBART
    AutoTokenizer,
)

# seed for random
seed = 42

lang_dict = {
    "en" : "English",
    "si" : "Sinhala",
    "ta" : "Tamil",
}

model_dict = {
    "mt5-base" : "google/mt5-base",
    "mbart-large-50" : "facebook/mbart-large-50",
    "m2m100_418M": "facebook/m2m100_418M",
    "indic-bartSS": "ai4bharat/IndicBARTSS",
}

tokenizer_lang_dic = {
    "en" : "en_XX",
    "si" : "si_LK",
    "ta" : "ta_IN",
    "dv" : "si_LK"
}

def get_model(model_name, weight_path = ""):

    if weight_path =="":
        model_path = model_dict[model_name]
    else:
        model_path = weight_path

    if(model_name == "mt5-base"):
        model = MT5ForConditionalGeneration.from_pretrained(model_path)

    elif(model_name == "mbart-large-50"):
        model = MBartForConditionalGeneration.from_pretrained(model_path)

    elif(model_name == "m2m100_418M"):
        model = M2M100ForConditionalGeneration.from_pretrained(model_path)

    elif(model_name == "indic-bartSS"):
        model = MBartForConditionalGeneration.from_pretrained(model_path)
        # Or use model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

    return model

def get_tokenizer(model_name, language = 'en', weight_path = ""):

    language_label = tokenizer_lang_dic[language]

    if weight_path =="":
        model_path = model_dict[model_name]
    else:
        model_path = weight_path

    if(model_name == "indic-bartSS"):
        tokenizer = AutoTokenizer.from_pretrained(model_path, do_lower_case=False, use_fast=False, keep_accents=True)
        # Or use tokenizer = AlbertTokenizer.from_pretrained(model_path, do_lower_case=False, use_fast=False, keep_accents=True)

    elif(model_name == "mt5-base"):
        tokenizer = MT5Tokenizer.from_pretrained(model_path)

    elif(model_name == "mbart-large-50"):
        tokenizer = MBart50Tokenizer.from_pretrained(model_path, src_lang=language_label, tgt_lang=language_label)

    elif(model_name == "m2m100_418M"):
        tokenizer = M2M100Tokenizer.from_pretrained(model_path)
        tokenizer.src_lang = language
        tokenizer.tgt_lang = language

    return tokenizer

Load Test Data to Evaluate

In [ ]:
import os
import torch
from tqdm import tqdm
from datasets import Dataset, load_metric
from google.cloud import storage
import gcsfs

# Config
EXPERIMENTS = ['I','J','G','H','K','L']
MODEL_NAME = 'mbart-large-50'

In [ ]:
model = get_model(MODEL_NAME)
tokenizer = get_tokenizer(MODEL_NAME, LANG)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def generate_MWP(model, tokenizer, text, lang=LANG_CODE, max_len=200):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(device)
    forced_bos_token_id = tokenizer.lang_code_to_id[lang]

    with torch.no_grad():
        generated_tokens = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_len,
            do_sample=True,
            temperature=1.0,
            num_return_sequences=1,
            forced_bos_token_id=forced_bos_token_id
        )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

def evaluate_metrics(reference_file, prediction_file):
    # Load BLEU
    bleu = load_metric("bleu")
    references = [[line.strip().split()] for line in open(reference_file, encoding="utf-8")]
    candidates = [line.strip().split() for line in open(prediction_file, encoding="utf-8")]
    bleu_result = bleu.compute(predictions=candidates, references=references, max_order=1)

    # Load METEOR
    meteor = load_metric("meteor")
    references_text = [line.strip() for line in open(reference_file, encoding="utf-8")]
    predictions_text = [line.strip() for line in open(prediction_file, encoding="utf-8")]
    meteor_result = meteor.compute(predictions=predictions_text, references=references_text)

    return round(bleu_result['bleu'], 4), round(meteor_result['meteor'], 4)

def process_experiment(exp_id):
    dataset_dir = f'{LANG}/{exp_id}'
    src_path = # your source path
    tgt_path = # your target path

    # Read data
    with fs.open(src_path, 'r') as f:
        test_sources = f.readlines()
    with fs.open(tgt_path, 'r') as f:
        test_targets = f.readlines()

    dataset = Dataset.from_dict({'source': test_sources, 'target': test_targets})

    output_file = f'{exp_id}_generated.txt'

    with open(output_file, "w", encoding="utf-8") as out_f:
        for idx in tqdm(range(len(dataset)), desc=f"Generating for {exp_id}"):
            source_text = dataset['source'][idx]
            gen_text = generate_MWP(model, tokenizer, source_text)
            out_f.write(gen_text.strip() + "\n")
            torch.cuda.empty_cache()

    # Evaluate
    bleu_score, meteor_score = evaluate_metrics(download_path, output_file)
    print(f"\n📊 {exp_id} — BLEU-1: {bleu_score}, METEOR: {meteor_score}")
